In [ ]:
!pip install -U bitsandbytes "transformers==4.57.1" accelerate timm einops zeus-ml -q

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", os.environ["HF_TOKEN"][:8], "...")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Visible devices:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "OpenGVLab/InternVL3_5-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    # NOTE: no attn_implementation="eager" here anymore — default (SDPA) for all layers
    device_map={"": 0},
).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
layers = model.language_model.model.layers

print("Loaded on:", next(model.parameters()).device)
print("GPU memory (GB):", torch.cuda.memory_allocated(0) / 1e9)
print("Default attn implementation:", model.language_model.config._attn_implementation)

In [ ]:
#5
layers = model.language_model.model.layers
print("Number of decoder layers:", len(layers))
print("Layer 0 type:", type(layers[0]))
print("Layer 0 self_attn type:", type(layers[0].self_attn))
print("Config attn_implementation:", model.language_model.config._attn_implementation)

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))

    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_path, input_size=448, max_num=6):
    image = Image.open(image_path).convert("RGB")
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    return torch.stack(pixel_values)

In [ ]:
#6
captured = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        attn_weights = output[1]
        if attn_weights is not None:
            captured[layer_idx] = attn_weights.detach().cpu()
        return (output[0], None) + output[2:]
    return hook

TEST_LAYERS = [3, 19]
handles = [layers[i].self_attn.register_forward_hook(make_hook(i)) for i in TEST_LAYERS]

from PIL import Image
import os

DATA_DIR = "/kaggle/input/notebooks/mmrafi14/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")
test_image_file = os.listdir(IMG_DIR)[0]

# TEMPORARY: max_num=2 instead of 6, just to test whether global eager fits
# at a shorter sequence length. NOT the real experimental setting.
pixel_values = load_image(os.path.join(IMG_DIR, test_image_file), max_num=2).to(torch.float16).cuda()
print("pixel_values shape:", pixel_values.shape)  # (num_tiles, 3, 448, 448)

question = "What is in this image? Answer the question using a single word or phrase."
generation_config = dict(max_new_tokens=32, do_sample=False, output_attentions=True)

with torch.no_grad():
    response = model.chat(tokenizer, pixel_values, question, generation_config)

print("Response:", response)
print("\nCaptured layers:", list(captured.keys()))
for idx, w in captured.items():
    print(f"Layer {idx} attention weights shape: {w.shape}")

print("\nGPU memory after capture (GB):", torch.cuda.memory_allocated(0) / 1e9)

for h in handles:
    h.remove()

In [ ]:
captured = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        attn_weights = output[1]
        if attn_weights is not None:
            captured[layer_idx] = attn_weights.detach().cpu()
        return (output[0], None) + output[2:]
    return hook

TEST_LAYERS = [3, 19]
handles = [layers[i].self_attn.register_forward_hook(make_hook(i)) for i in TEST_LAYERS]

# --- Intercept the real embedding merge InternVL performs internally ---
# InternVLChatModel.chat() -> self.generate() -> self.language_model.generate(inputs_embeds=..., attention_mask=...)
# By the time language_model.generate() is called, InternVL has ALREADY done the
# extract_feature() + vit_embeds-scatter-into-input_embeds merge correctly.
# We intercept right there instead of reimplementing that merge ourselves.
intercepted = {}
original_lm_generate = model.language_model.generate

def spy_generate(*args, **kwargs):
    intercepted['inputs_embeds'] = kwargs.get('inputs_embeds')
    intercepted['attention_mask'] = kwargs.get('attention_mask')
    raise RuntimeError("INTERCEPTED_OK")  # deliberate early stop, not a real error

model.language_model.generate = spy_generate

pixel_values = load_image(os.path.join(IMG_DIR, test_image_file), max_num=2).to(torch.float16).cuda()
question = "What is in this image? Answer the question using a single word or phrase."
generation_config = dict(max_new_tokens=32, do_sample=False)

try:
    with torch.no_grad():
        _ = model.chat(tokenizer, pixel_values, question, generation_config)
except RuntimeError as e:
    if "INTERCEPTED_OK" not in str(e):
        raise  # a real error — don't swallow it

# restore the real generate method immediately
model.language_model.generate = original_lm_generate

print("Intercepted inputs_embeds shape:", intercepted['inputs_embeds'].shape)
print("Intercepted attention_mask shape:", intercepted['attention_mask'].shape)

# --- Now run ONE real forward pass ourselves (prefill only, no autoregressive loop) ---
with torch.no_grad():
    outputs = model.language_model(
        inputs_embeds=intercepted['inputs_embeds'],
        attention_mask=intercepted['attention_mask'],
        output_attentions=True,
        use_cache=False,
    )

print("\nCaptured layers:", list(captured.keys()))
for idx, w in captured.items():
    print(f"Layer {idx} attention weights shape: {w.shape}")

print("\nGPU memory after single forward (GB):", torch.cuda.memory_allocated(0) / 1e9)

for h in handles:
    h.remove()

In [ ]:
import types
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb, eager_attention_forward

captured = {}

def make_patched_forward(layer_idx):
    def patched_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        key_states = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        # Force eager computation for THIS layer only, regardless of global config
        attn_output, attn_weights = eager_attention_forward(
            self, query_states, key_states, value_states, attention_mask,
            dropout=0.0,
            scaling=self.scaling,
        )
        captured[layer_idx] = attn_weights.detach().cpu()

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        return attn_output, None  # None matches what SDPA layers return — decoder layer discards it anyway

    return patched_forward

TARGET_LAYERS = [3, 19]
original_forwards = {}
for idx in TARGET_LAYERS:
    attn_module = layers[idx].self_attn
    original_forwards[idx] = attn_module.forward
    attn_module.forward = types.MethodType(make_patched_forward(idx), attn_module)

print(f"Patched layers: {TARGET_LAYERS}")

In [ ]:
#9
intercepted = {}
original_lm_generate = model.language_model.generate

def spy_generate(*args, **kwargs):
    intercepted['inputs_embeds'] = kwargs.get('inputs_embeds')
    intercepted['attention_mask'] = kwargs.get('attention_mask')
    raise RuntimeError("INTERCEPTED_OK")

model.language_model.generate = spy_generate

# REAL experimental setting now — max_num=6, matching the baseline notebook
pixel_values = load_image(os.path.join(IMG_DIR, test_image_file), max_num=6).to(torch.float16).cuda()
question = "What is in this image? Answer the question using a single word or phrase."
generation_config = dict(max_new_tokens=32, do_sample=False)

try:
    with torch.no_grad():
        _ = model.chat(tokenizer, pixel_values, question, generation_config)
except RuntimeError as e:
    if "INTERCEPTED_OK" not in str(e):
        raise

model.language_model.generate = original_lm_generate

print("inputs_embeds shape:", intercepted['inputs_embeds'].shape)

with torch.no_grad():
    outputs = model.language_model(
        inputs_embeds=intercepted['inputs_embeds'],
        attention_mask=intercepted['attention_mask'],
        use_cache=False,
    )

print("\nCaptured layers:", list(captured.keys()))
for idx, w in captured.items():
    print(f"Layer {idx} attention weights shape: {w.shape}")

print("\nGPU memory after full-scale forward (GB):", torch.cuda.memory_allocated(0) / 1e9)

# restore original forwards
for idx in TARGET_LAYERS:
    layers[idx].self_attn.forward = original_forwards[idx]

In [ ]:
# --- Run 1: baseline, fully unpatched ---
pixel_values = load_image(os.path.join(IMG_DIR, test_image_file), max_num=6).to(torch.float16).cuda()
question = "What is in this image? Answer the question using a single word or phrase."
generation_config = dict(max_new_tokens=32, do_sample=False)

with torch.no_grad():
    baseline_response = model.chat(tokenizer, pixel_values, question, generation_config)
print("Baseline (unpatched) response:", repr(baseline_response))

# --- Apply the patch (same as Cell 8) ---
captured = {}
TARGET_LAYERS = [3, 19]
original_forwards = {}
for idx in TARGET_LAYERS:
    attn_module = layers[idx].self_attn
    original_forwards[idx] = attn_module.forward
    attn_module.forward = types.MethodType(make_patched_forward(idx), attn_module)

# --- Run 2: patched (eager capture on layers 3, 19; SDPA elsewhere) ---
with torch.no_grad():
    patched_response = model.chat(tokenizer, pixel_values, question, generation_config)
print("Patched response:", repr(patched_response))

print("\nMatch:", baseline_response == patched_response)
print("Captured layers during patched run:", list(captured.keys()))
for idx, w in captured.items():
    print(f"Layer {idx} shape: {w.shape}")

# restore for cleanliness
for idx in TARGET_LAYERS:
    layers[idx].self_attn.forward = original_forwards[idx]

In [ ]:
#11
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask

def build_mask_and_positions(model, inputs_embeds, attention_mask, past_key_values=None):
    lm = model.language_model.model  # the Qwen3Model instance
    config = lm.config
    device = inputs_embeds.device
    seq_len = inputs_embeds.shape[1]

    past_seen = past_key_values.get_seq_length() if past_key_values is not None else 0
    cache_position = torch.arange(past_seen, past_seen + seq_len, device=device)
    position_ids = cache_position.unsqueeze(0)

    mask_kwargs = {
        "config": config,
        "input_embeds": inputs_embeds,          # renamed from inputs_embeds
        "attention_mask": attention_mask,
        "cache_position": cache_position,        # newly required
        "past_key_values": past_key_values,
        "position_ids": position_ids,
    }
    causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
    if getattr(lm, "has_sliding_layers", False):
        causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**mask_kwargs)

    position_embeddings = lm.rotary_emb(inputs_embeds, position_ids)
    return causal_mask_mapping, position_embeddings, position_ids

In [ ]:
#12
def dap_forward_v0(model, inputs_embeds, attention_mask):
    """
    0%-pruning scaffold. Manually loops over decoder layers instead of calling
    Qwen3Model.forward() directly — this loop is where checkpoint-based pruning
    will be inserted later. For now: must match stock output exactly.
    """
    lm = model.language_model.model
    layers = lm.layers
    config = lm.config

    past_key_values = DynamicCache(config=config)
    causal_mask_mapping, position_embeddings, position_ids = build_mask_and_positions(
        model, inputs_embeds, attention_mask, past_key_values=None
    )

    hidden_states = inputs_embeds
    for i, decoder_layer in enumerate(layers[: config.num_hidden_layers]):
        hidden_states = decoder_layer(
            hidden_states,
            attention_mask=causal_mask_mapping[config.layer_types[i]],
            position_embeddings=position_embeddings,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=True,
        )

    hidden_states = lm.norm(hidden_states)
    return hidden_states, past_key_values

In [ ]:
# Reuse the same intercepted inputs_embeds/attention_mask from Cell 9
inputs_embeds = intercepted['inputs_embeds']
attention_mask = intercepted['attention_mask']

# --- Path A: stock, built-in Qwen3Model.forward ---
with torch.no_grad():
    stock_out = model.language_model.model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        use_cache=False,
    )
stock_hidden = stock_out.last_hidden_state

# --- Path B: our manual dap_forward_v0 ---
with torch.no_grad():
    manual_hidden, _ = dap_forward_v0(model, inputs_embeds, attention_mask)

max_diff = (stock_hidden - manual_hidden).abs().max().item()
print("Stock hidden_states shape:", stock_hidden.shape)
print("Manual hidden_states shape:", manual_hidden.shape)
print("Max absolute difference:", max_diff)
print("Byte-identical (max_diff == 0.0):", max_diff == 0.0)

In [ ]:
# InternVL sets self.img_context_token_id internally during chat() — capture input_ids
# by spying one level higher, at the outer model.generate call (not language_model.generate)
intercepted_ids = {}
original_model_generate = model.generate

def spy_model_generate(*args, **kwargs):
    intercepted_ids['input_ids'] = kwargs.get('input_ids')
    intercepted_ids['img_context_token_id'] = getattr(model, 'img_context_token_id', None)
    raise RuntimeError("INTERCEPTED_IDS_OK")

model.generate = spy_model_generate

pixel_values = load_image(os.path.join(IMG_DIR, test_image_file), max_num=6).to(torch.float16).cuda()
question = "What is in this image? Answer the question using a single word or phrase."
generation_config = dict(max_new_tokens=32, do_sample=False)

try:
    with torch.no_grad():
        _ = model.chat(tokenizer, pixel_values, question, generation_config)
except RuntimeError as e:
    if "INTERCEPTED_IDS_OK" not in str(e):
        raise

model.generate = original_model_generate

input_ids = intercepted_ids['input_ids']
img_context_token_id = intercepted_ids['img_context_token_id']
print("input_ids shape:", input_ids.shape)
print("img_context_token_id:", img_context_token_id)

visual_mask = (input_ids == img_context_token_id).squeeze(0)  # True at visual token positions
visual_indices = torch.nonzero(visual_mask, as_tuple=True)[0]
print("Number of visual tokens:", visual_indices.numel())
print("Visual token index range:", visual_indices.min().item(), "-", visual_indices.max().item())
print("Total sequence length:", input_ids.shape[1])

In [ ]:
#15
def compute_mean_importance(attn_weights, visual_indices):
    """
    attn_weights: [1, num_heads, seq, seq] from one eager layer
    visual_indices: 1D tensor of candidate (visual) token positions
    Returns: importance score per visual token — mean attention RECEIVED,
    averaged over heads and over causally-valid query positions (i >= j).
    """
    device = attn_weights.device
    seq_len = attn_weights.shape[-1]
    aw = attn_weights[0].mean(dim=0)  # average over heads -> [seq, seq] (query, key)

    seq_range = torch.arange(seq_len, device=device)
    valid_counts = seq_len - seq_range  # for key j, valid queries are i in [j, seq_len-1] -> count = seq_len - j

    col_sums = aw.sum(dim=0)  # sum over query dim for each key -> [seq]
    importance_all = col_sums / valid_counts.clamp(min=1).to(col_sums.dtype)

    return importance_all[visual_indices.to(device)]  # ensure visual_indices matches device too


def dap_forward_ckpt1(model, inputs_embeds, attention_mask, visual_indices, checkpoint_layer=10, keep_pctile=50):
    lm = model.language_model.model
    layers = lm.layers
    config = lm.config

    causal_mask_mapping, position_embeddings, position_ids = build_mask_and_positions(
        model, inputs_embeds, attention_mask, past_key_values=None
    )

    hidden_states = inputs_embeds
    seq_len = hidden_states.shape[1]
    all_indices = torch.arange(seq_len, device=hidden_states.device)

    captured_scores = {}

    for i, decoder_layer in enumerate(layers[: config.num_hidden_layers]):
        if i == checkpoint_layer:
            # Temporarily patch THIS layer to eager + capture, for THIS forward only
            attn_module = decoder_layer.self_attn
            original_fwd = attn_module.forward
            local_capture = {}

            def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                input_shape = hidden_states.shape[:-1]
                hidden_shape = (*input_shape, -1, self.head_dim)
                q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                cos, sin = position_embeddings
                q, k = apply_rotary_pos_emb(q, k, cos, sin)
                attn_output, attn_weights = eager_attention_forward(
                    self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling
                )
                local_capture['weights'] = attn_weights.detach()
                attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                attn_output = self.o_proj(attn_output)
                return attn_output, None

            attn_module.forward = types.MethodType(scoring_forward, attn_module)

            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings,
                position_ids=position_ids,
                past_key_values=None,
                use_cache=False,
            )

            attn_module.forward = original_fwd  # restore immediately

            # --- Score and select survivors ---
            importance = compute_mean_importance(local_capture['weights'], visual_indices)
            cutoff = torch.quantile(importance.float(), keep_pctile / 100.0)
            keep_visual_mask = importance >= cutoff
            dropped_visual = visual_indices[~keep_visual_mask]
            kept_visual = visual_indices[keep_visual_mask]

            captured_scores['importance'] = importance
            captured_scores['dropped_count'] = dropped_visual.numel()
            captured_scores['kept_count'] = kept_visual.numel()

            # Build surviving index set: all non-visual (text) tokens + kept visual tokens
            is_visual = torch.zeros(seq_len, dtype=torch.bool, device=hidden_states.device)
            is_visual[visual_indices] = True
            text_indices = all_indices[~is_visual]
            surviving_indices = torch.sort(torch.cat([text_indices, kept_visual])).values

            # Slice hidden_states to survivors only
            hidden_states = hidden_states[:, surviving_indices, :]
            new_seq_len = hidden_states.shape[1]

            # Rebuild mask + position embeddings for the NEW shorter sequence
            new_attention_mask = torch.ones(1, new_seq_len, device=hidden_states.device, dtype=attention_mask.dtype)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_and_positions(
                model, hidden_states, new_attention_mask, past_key_values=None
            )
            captured_scores['surviving_indices'] = surviving_indices

        else:
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings,
                position_ids=position_ids,
                past_key_values=None,
                use_cache=False,
            )

    hidden_states = lm.norm(hidden_states)
    return hidden_states, captured_scores

In [ ]:
inputs_embeds = intercepted['inputs_embeds']
attention_mask_full = intercepted['attention_mask']

# --- Unpruned baseline: reuse dap_forward_v0 from Cell 12 ---
with torch.no_grad():
    unpruned_hidden, _ = dap_forward_v0(model, inputs_embeds, attention_mask_full)
    unpruned_logits = model.language_model.lm_head(unpruned_hidden[:, -1:, :])
    unpruned_next_token = unpruned_logits.argmax(dim=-1)

# --- Pruned: checkpoint-1 at layer 10, keep top 50% of visual tokens by mean importance ---
with torch.no_grad():
    pruned_hidden, scores = dap_forward_ckpt1(
        model, inputs_embeds, attention_mask_full, visual_indices,
        checkpoint_layer=10, keep_pctile=50
    )
    pruned_logits = model.language_model.lm_head(pruned_hidden[:, -1:, :])
    pruned_next_token = pruned_logits.argmax(dim=-1)

print("Visual tokens dropped:", scores['dropped_count'], "/ kept:", scores['kept_count'])
print("Unpruned next token id:", unpruned_next_token.item(), "->", tokenizer.decode(unpruned_next_token[0]))
print("Pruned next token id:  ", pruned_next_token.item(), "->", tokenizer.decode(pruned_next_token[0]))
print("Match:", unpruned_next_token.item() == pruned_next_token.item())

In [ ]:
test_files = os.listdir(IMG_DIR)[:5]

results = []
for fname in test_files:
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()
    question = "What is in this image? Answer the question using a single word or phrase."
    generation_config = dict(max_new_tokens=32, do_sample=False)

    # capture embeds + input_ids for this sample
    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        intercepted_local['attention_mask'] = kwargs.get('attention_mask')
        raise RuntimeError("OK1")
    model.language_model.generate = spy_lm_generate
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, generation_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = original_lm_generate

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, generation_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = original_model_generate

    ie = intercepted_local['inputs_embeds']
    am = intercepted_local['attention_mask']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == img_context_token_id).squeeze(0)
    vis_idx = torch.nonzero(vis_mask, as_tuple=True)[0]

    with torch.no_grad():
        unpruned_hidden, _ = dap_forward_v0(model, ie, am)
        unpruned_tok = model.language_model.lm_head(unpruned_hidden[:, -1:, :]).argmax(dim=-1).item()

        pruned_hidden, scores = dap_forward_ckpt1(model, ie, am, vis_idx, checkpoint_layer=10, keep_pctile=50)
        pruned_tok = model.language_model.lm_head(pruned_hidden[:, -1:, :]).argmax(dim=-1).item()

    match = unpruned_tok == pruned_tok
    results.append({
        "file": fname, "match": match,
        "unpruned": tokenizer.decode([unpruned_tok]),
        "pruned": tokenizer.decode([pruned_tok]),
        "dropped": scores['dropped_count'], "kept": scores['kept_count'],
    })
    print(f"{fname}: match={match} | unpruned='{tokenizer.decode([unpruned_tok])}' pruned='{tokenizer.decode([pruned_tok])}' | dropped={scores['dropped_count']}")

match_rate = sum(r['match'] for r in results) / len(results)
print(f"\nMatch rate: {match_rate*100:.0f}% ({sum(r['match'] for r in results)}/{len(results)})")

In [ ]:
#18
import json

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

TEST_SAMPLES = SAMPLES[:40]
print(f"Testing on {len(TEST_SAMPLES)} real VQA samples")

def vqa_accuracy(predicted: str, ground_truth_answers: list) -> float:
    import re
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for article in ["a ", "an ", "the "]:
            if s.startswith(article):
                s = s[len(article):]
        return s.strip()
    pred_norm = normalise(predicted)
    matches = sum(1 for a in ground_truth_answers if normalise(a) == pred_norm)
    return min(matches / 3, 1.0)

In [ ]:
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."

results = []
for s in TEST_SAMPLES:
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()
    generation_config = dict(max_new_tokens=32, do_sample=False)

    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        intercepted_local['attention_mask'] = kwargs.get('attention_mask')
        raise RuntimeError("OK1")
    model.language_model.generate = spy_lm_generate
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = original_lm_generate

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = original_model_generate

    ie = intercepted_local['inputs_embeds']
    am = intercepted_local['attention_mask']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == img_context_token_id).squeeze(0)
    vis_idx = torch.nonzero(vis_mask, as_tuple=True)[0]

    with torch.no_grad():
        unpruned_hidden, _ = dap_forward_v0(model, ie, am)
        unpruned_tok_id = model.language_model.lm_head(unpruned_hidden[:, -1:, :]).argmax(dim=-1).item()

        pruned_hidden, scores = dap_forward_ckpt1(model, ie, am, vis_idx, checkpoint_layer=10, keep_pctile=50)
        pruned_tok_id = model.language_model.lm_head(pruned_hidden[:, -1:, :]).argmax(dim=-1).item()

    unpruned_text = tokenizer.decode([unpruned_tok_id]).strip()
    pruned_text = tokenizer.decode([pruned_tok_id]).strip()

    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    pruned_acc = vqa_accuracy(pruned_text, s["answers"])
    token_match = unpruned_tok_id == pruned_tok_id

    results.append({
        "file": fname, "question": s["question"], "token_match": token_match,
        "unpruned_text": unpruned_text, "pruned_text": pruned_text,
        "unpruned_acc": unpruned_acc, "pruned_acc": pruned_acc,
        "dropped": scores['dropped_count'],
    })

n = len(results)
token_match_rate = sum(r['token_match'] for r in results) / n
avg_unpruned_acc = sum(r['unpruned_acc'] for r in results) / n
avg_pruned_acc = sum(r['pruned_acc'] for r in results) / n

print(f"n = {n}")
print(f"Token match rate: {token_match_rate*100:.1f}%")
print(f"Avg unpruned accuracy: {avg_unpruned_acc:.4f}")
print(f"Avg pruned accuracy:   {avg_pruned_acc:.4f}")
print(f"Accuracy delta (pruned - unpruned): {avg_pruned_acc - avg_unpruned_acc:+.4f}")

# show any mismatches for inspection
mismatches = [r for r in results if not r['token_match']]
print(f"\n{len(mismatches)} mismatches:")
for r in mismatches:
    print(f"  {r['file']}: Q='{r['question']}' | unpruned='{r['unpruned_text']}' pruned='{r['pruned_text']}' | unpruned_acc={r['unpruned_acc']:.2f} pruned_acc={r['pruned_acc']:.2f}")

In [ ]:
DROP_RATIOS = [0.25, 0.50, 0.75]  # fraction of visual tokens dropped at checkpoint 1

stress_results = {ratio: [] for ratio in DROP_RATIOS}
unpruned_cache = []

for s in TEST_SAMPLES:
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()
    generation_config = dict(max_new_tokens=32, do_sample=False)

    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        intercepted_local['attention_mask'] = kwargs.get('attention_mask')
        raise RuntimeError("OK1")
    model.language_model.generate = spy_lm_generate
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = original_lm_generate

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = original_model_generate

    ie = intercepted_local['inputs_embeds']
    am = intercepted_local['attention_mask']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == img_context_token_id).squeeze(0)
    vis_idx = torch.nonzero(vis_mask, as_tuple=True)[0]

    with torch.no_grad():
        unpruned_hidden, _ = dap_forward_v0(model, ie, am)
        unpruned_tok_id = model.language_model.lm_head(unpruned_hidden[:, -1:, :]).argmax(dim=-1).item()
    unpruned_text = tokenizer.decode([unpruned_tok_id]).strip()
    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    unpruned_cache.append(unpruned_acc)

    for ratio in DROP_RATIOS:
        keep_pctile = ratio * 100  # dap_forward_ckpt1's keep_pctile IS the drop percentile cutoff
        with torch.no_grad():
            pruned_hidden, scores = dap_forward_ckpt1(model, ie, am, vis_idx, checkpoint_layer=10, keep_pctile=keep_pctile)
            pruned_tok_id = model.language_model.lm_head(pruned_hidden[:, -1:, :]).argmax(dim=-1).item()
        pruned_text = tokenizer.decode([pruned_tok_id]).strip()
        pruned_acc = vqa_accuracy(pruned_text, s["answers"])
        stress_results[ratio].append(pruned_acc)

avg_unpruned = sum(unpruned_cache) / len(unpruned_cache)
print(f"Baseline (unpruned) accuracy: {avg_unpruned:.4f}\n")
print(f"{'Drop Ratio':<12}{'Avg Accuracy':<15}{'Delta':<10}")
for ratio in DROP_RATIOS:
    avg_acc = sum(stress_results[ratio]) / len(stress_results[ratio])
    delta = avg_acc - avg_unpruned
    print(f"{ratio*100:>5.0f}%       {avg_acc:<15.4f}{delta:+.4f}")

In [ ]:
#21
CHECKPOINT_LAYERS = [10, 14, 18]  # rescaled per §3.4
KEEP_PCTILE = 70   # >= this percentile of current candidates: keep outright
DROP_PCTILE = 30   # <= this percentile: drop outright; middle band defers

def dap_forward_v1(model, inputs_embeds, attention_mask, visual_indices,
                    checkpoint_layers=CHECKPOINT_LAYERS, keep_pctile=KEEP_PCTILE, drop_pctile=DROP_PCTILE):
    lm = model.language_model.model
    layers = lm.layers
    config = lm.config

    causal_mask_mapping, position_embeddings, position_ids = build_mask_and_positions(
        model, inputs_embeds, attention_mask, past_key_values=None
    )

    hidden_states = inputs_embeds
    deferred_local = visual_indices.clone().to(hidden_states.device)  # current-sequence-space positions
    log = {"checkpoints": []}

    for i, decoder_layer in enumerate(layers[: config.num_hidden_layers]):
        if i in checkpoint_layers:
            is_final = (i == checkpoint_layers[-1])

            attn_module = decoder_layer.self_attn
            original_fwd = attn_module.forward
            local_capture = {}

            def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                input_shape = hidden_states.shape[:-1]
                hidden_shape = (*input_shape, -1, self.head_dim)
                q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                cos, sin = position_embeddings
                q, k = apply_rotary_pos_emb(q, k, cos, sin)
                attn_output, attn_weights = eager_attention_forward(
                    self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling
                )
                local_capture['weights'] = attn_weights.detach()
                attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                attn_output = self.o_proj(attn_output)
                return attn_output, None

            attn_module.forward = types.MethodType(scoring_forward, attn_module)
            hidden_states = decoder_layer(
                hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings, position_ids=position_ids,
                past_key_values=None, use_cache=False,
            )
            attn_module.forward = original_fwd

            if deferred_local.numel() == 0:
                log["checkpoints"].append({"layer": i, "n_candidates": 0, "kept": 0, "dropped": 0, "deferred": 0})
                continue

            importance = compute_mean_importance(local_capture['weights'], deferred_local)

            if is_final:
                # forced resolution: no defer option, split remaining pool at its own median
                median_cut = torch.median(importance.float())
                keep_mask = importance >= median_cut
                keep_idx = deferred_local[keep_mask]
                drop_idx = deferred_local[~keep_mask]
                new_deferred = torch.tensor([], dtype=torch.long, device=hidden_states.device)
            else:
                keep_cut = torch.quantile(importance.float(), keep_pctile / 100.0)
                drop_cut = torch.quantile(importance.float(), drop_pctile / 100.0)
                keep_mask = importance >= keep_cut
                drop_mask = (importance <= drop_cut) & (~keep_mask)
                defer_mask = (~keep_mask) & (~drop_mask)
                keep_idx = deferred_local[keep_mask]
                drop_idx = deferred_local[drop_mask]
                new_deferred = deferred_local[defer_mask]

            log["checkpoints"].append({
                "layer": i, "n_candidates": deferred_local.numel(),
                "kept": keep_idx.numel(), "dropped": drop_idx.numel(), "deferred": new_deferred.numel(),
            })

            # --- Bug-3 fix: remap ALL surviving indices into the new post-drop coordinate space ---
            cur_seq_len = hidden_states.shape[1]
            all_idx = torch.arange(cur_seq_len, device=hidden_states.device)
            is_candidate = torch.zeros(cur_seq_len, dtype=torch.bool, device=hidden_states.device)
            is_candidate[deferred_local] = True
            non_candidate_idx = all_idx[~is_candidate]  # text tokens + visual tokens kept at an EARLIER checkpoint

            surviving = torch.sort(torch.cat([non_candidate_idx, keep_idx, new_deferred])).values
            remap = torch.full((cur_seq_len,), -1, dtype=torch.long, device=hidden_states.device)
            remap[surviving] = torch.arange(surviving.numel(), device=hidden_states.device)

            hidden_states = hidden_states[:, surviving, :]
            deferred_local = remap[new_deferred] if new_deferred.numel() > 0 else new_deferred

            new_attention_mask = torch.ones(1, hidden_states.shape[1], device=hidden_states.device, dtype=attention_mask.dtype)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_and_positions(
                model, hidden_states, new_attention_mask, past_key_values=None
            )
        else:
            hidden_states = decoder_layer(
                hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings, position_ids=position_ids,
                past_key_values=None, use_cache=False,
            )

    hidden_states = lm.norm(hidden_states)
    return hidden_states, log

In [ ]:
#22
door_sample = next(s for s in TEST_SAMPLES if s["image_file"] == "554523026.jpg")
question_text = f"{door_sample['question']} {VQA_PROMPT_SUFFIX}"
pixel_values = load_image(os.path.join(IMG_DIR, door_sample["image_file"]), max_num=6).to(torch.float16).cuda()
generation_config = dict(max_new_tokens=32, do_sample=False)

intercepted_local = {}
def spy_lm_generate(*args, **kwargs):
    intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
    intercepted_local['attention_mask'] = kwargs.get('attention_mask')
    raise RuntimeError("OK1")
model.language_model.generate = spy_lm_generate
try:
    with torch.no_grad():
        _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
except RuntimeError as e:
    if "OK1" not in str(e): raise
model.language_model.generate = original_lm_generate

intercepted_ids_local = {}
def spy_model_gen(*args, **kwargs):
    intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
    raise RuntimeError("OK2")
model.generate = spy_model_gen
try:
    with torch.no_grad():
        _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
except RuntimeError as e:
    if "OK2" not in str(e): raise
model.generate = original_model_generate

ie = intercepted_local['inputs_embeds']
am = intercepted_local['attention_mask']
ids = intercepted_ids_local['input_ids']
vis_mask = (ids == img_context_token_id).squeeze(0)
vis_idx = torch.nonzero(vis_mask, as_tuple=True)[0]

with torch.no_grad():
    dap_hidden, log = dap_forward_v1(model, ie, am, vis_idx)
    dap_tok = model.language_model.lm_head(dap_hidden[:, -1:, :]).argmax(dim=-1).item()

print("Question:", door_sample["question"])
print("Ground truth answers:", door_sample["answers"])
print("DAP (3-checkpoint) prediction:", tokenizer.decode([dap_tok]).strip())
print("\nCheckpoint log:")
for c in log["checkpoints"]:
    print(f"  layer {c['layer']}: candidates={c['n_candidates']} kept={c['kept']} dropped={c['dropped']} deferred={c['deferred']}")

In [ ]:
comparison_results = []

for s in TEST_SAMPLES:
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()
    generation_config = dict(max_new_tokens=32, do_sample=False)

    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        intercepted_local['attention_mask'] = kwargs.get('attention_mask')
        raise RuntimeError("OK1")
    model.language_model.generate = spy_lm_generate
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = original_lm_generate

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question_text, generation_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = original_model_generate

    ie = intercepted_local['inputs_embeds']
    am = intercepted_local['attention_mask']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == img_context_token_id).squeeze(0)
    vis_idx = torch.nonzero(vis_mask, as_tuple=True)[0]

    with torch.no_grad():
        # Unpruned
        unpruned_hidden, _ = dap_forward_v0(model, ie, am)
        unpruned_tok = model.language_model.lm_head(unpruned_hidden[:, -1:, :]).argmax(dim=-1).item()

        # Checkpoint-1-only (50% drop, same as Cell 19)
        ckpt1_hidden, ckpt1_scores = dap_forward_ckpt1(model, ie, am, vis_idx, checkpoint_layer=10, keep_pctile=50)
        ckpt1_tok = model.language_model.lm_head(ckpt1_hidden[:, -1:, :]).argmax(dim=-1).item()

        # Full 3-checkpoint DAP
        dap_hidden, dap_log = dap_forward_v1(model, ie, am, vis_idx)
        dap_tok = model.language_model.lm_head(dap_hidden[:, -1:, :]).argmax(dim=-1).item()

    unpruned_text = tokenizer.decode([unpruned_tok]).strip()
    ckpt1_text = tokenizer.decode([ckpt1_tok]).strip()
    dap_text = tokenizer.decode([dap_tok]).strip()

    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    ckpt1_acc = vqa_accuracy(ckpt1_text, s["answers"])
    dap_acc = vqa_accuracy(dap_text, s["answers"])

    final_dropped = sum(c['dropped'] for c in dap_log['checkpoints'])

    comparison_results.append({
        "file": fname, "question": s["question"],
        "unpruned_text": unpruned_text, "ckpt1_text": ckpt1_text, "dap_text": dap_text,
        "unpruned_acc": unpruned_acc, "ckpt1_acc": ckpt1_acc, "dap_acc": dap_acc,
        "ckpt1_dropped": ckpt1_scores['dropped_count'], "dap_total_dropped": final_dropped,
    })

n = len(comparison_results)
avg_unpruned = sum(r['unpruned_acc'] for r in comparison_results) / n
avg_ckpt1 = sum(r['ckpt1_acc'] for r in comparison_results) / n
avg_dap = sum(r['dap_acc'] for r in comparison_results) / n

print(f"n = {n}\n")
print(f"{'Method':<25}{'Avg Accuracy':<15}{'Delta vs unpruned':<20}")
print(f"{'Unpruned baseline':<25}{avg_unpruned:<15.4f}{'—':<20}")
print(f"{'Checkpoint-1-only':<25}{avg_ckpt1:<15.4f}{avg_ckpt1-avg_unpruned:+.4f}")
print(f"{'Full 3-checkpoint DAP':<25}{avg_dap:<15.4f}{avg_dap-avg_unpruned:+.4f}")

print(f"\nAvg tokens dropped (checkpoint-1-only): {sum(r['ckpt1_dropped'] for r in comparison_results)/n:.1f}")
print(f"Avg tokens dropped (full DAP): {sum(r['dap_total_dropped'] for r in comparison_results)/n:.1f}")

print("\nPer-sample where DAP and checkpoint-1-only disagree with each other or with unpruned:")
for r in comparison_results:
    if not (r['unpruned_text'] == r['ckpt1_text'] == r['dap_text']):
        print(f"  {r['file']}: Q='{r['question']}'")
        print(f"    unpruned='{r['unpruned_text']}' (acc={r['unpruned_acc']:.2f})  "
              f"ckpt1-only='{r['ckpt1_text']}' (acc={r['ckpt1_acc']:.2f})  "
              f"full-DAP='{r['dap_text']}' (acc={r['dap_acc']:.2f})")

In [ ]:
#24
from transformers.cache_utils import DynamicCache
import inspect

dummy_cache = DynamicCache(config=model.language_model.config)
print("DynamicCache attributes:", [a for a in dir(dummy_cache) if not a.startswith('_')])
print()
print(inspect.signature(dummy_cache.update))
print()
print(inspect.getsource(dummy_cache.update))

In [ ]:
#24b
# Trigger creation of one layer's cache entry so we can inspect its real structure
dummy_k = torch.zeros(1, 2, 5, 8, device=model.device, dtype=torch.float16)
dummy_v = torch.zeros(1, 2, 5, 8, device=model.device, dtype=torch.float16)
dummy_cache.update(dummy_k, dummy_v, layer_idx=0)

layer_obj = dummy_cache.layers[0]
print("Layer object type:", type(layer_obj))
print("Layer object attributes:", [a for a in dir(layer_obj) if not a.startswith('_')])
print()
print("keys shape:", layer_obj.keys.shape if hasattr(layer_obj, 'keys') else "no .keys attr")
print("values shape:", layer_obj.values.shape if hasattr(layer_obj, 'values') else "no .values attr")

In [ ]:
#25
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask

def dap_generate(model, tokenizer, pixel_values, question, max_new_tokens=32,
                  checkpoint_layers=CHECKPOINT_LAYERS, keep_pctile=KEEP_PCTILE, drop_pctile=DROP_PCTILE):
    device = next(model.parameters()).device
    lm = model.language_model.model
    layers = lm.layers
    config = lm.config

    # --- Step 1: get real merged embeddings + input_ids via intercept ---
    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        raise RuntimeError("OK1")
    orig_lm_gen = model.language_model.generate
    model.language_model.generate = spy_lm_generate
    probe_config = dict(max_new_tokens=1, do_sample=False)
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = orig_lm_gen

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    orig_model_gen = model.generate
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = orig_model_gen

    inputs_embeds = intercepted_local['inputs_embeds']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == model.img_context_token_id).squeeze(0)
    visual_indices = torch.nonzero(vis_mask, as_tuple=True)[0].to(device)

    def build_mask_pos(h, attn_mask_1d, cache_position, past_kv):
        pos_ids = cache_position.unsqueeze(0)
        mask_kwargs = {
            "config": config, "input_embeds": h, "attention_mask": attn_mask_1d,
            "cache_position": cache_position, "past_key_values": past_kv, "position_ids": pos_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        pos_emb = lm.rotary_emb(h, pos_ids)
        return causal_mask_mapping, pos_emb, pos_ids

    # --- Step 2: prefill with progressive checkpoint-based pruning ---
    past_key_values = DynamicCache(config=config)
    hidden_states = inputs_embeds
    deferred_local = visual_indices.clone()

    for i, decoder_layer in enumerate(layers[: config.num_hidden_layers]):
        cur_len = hidden_states.shape[1]
        cache_position = torch.arange(0, cur_len, device=device)  # prefill: this layer's cache is empty so far
        attn_mask_1d = torch.ones(1, cur_len, device=device, dtype=torch.long)
        causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(
            hidden_states, attn_mask_1d, cache_position, past_key_values
        )

        if i in checkpoint_layers:
            is_final = (i == checkpoint_layers[-1])
            attn_module = decoder_layer.self_attn
            original_fwd = attn_module.forward
            local_capture = {}

            def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                input_shape = hidden_states.shape[:-1]
                hidden_shape = (*input_shape, -1, self.head_dim)
                q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                cos, sin = position_embeddings
                q, k = apply_rotary_pos_emb(q, k, cos, sin)
                if past_key_values is not None:
                    k, v = past_key_values.update(k, v, self.layer_idx)
                attn_output, attn_weights = eager_attention_forward(self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling)
                local_capture['weights'] = attn_weights.detach()
                attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                attn_output = self.o_proj(attn_output)
                return attn_output, None

            attn_module.forward = types.MethodType(scoring_forward, attn_module)
            hidden_states = decoder_layer(
                hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings, position_ids=position_ids,
                past_key_values=past_key_values, use_cache=True,
            )
            attn_module.forward = original_fwd

            if deferred_local.numel() > 0:
                importance = compute_mean_importance(local_capture['weights'], deferred_local)
                if is_final:
                    median_cut = torch.median(importance.float())
                    keep_mask = importance >= median_cut
                    keep_idx = deferred_local[keep_mask]; drop_idx = deferred_local[~keep_mask]
                    new_deferred = torch.tensor([], dtype=torch.long, device=device)
                else:
                    keep_cut = torch.quantile(importance.float(), keep_pctile / 100.0)
                    drop_cut = torch.quantile(importance.float(), drop_pctile / 100.0)
                    keep_mask = importance >= keep_cut
                    drop_mask = (importance <= drop_cut) & (~keep_mask)
                    defer_mask = (~keep_mask) & (~drop_mask)
                    keep_idx = deferred_local[keep_mask]; drop_idx = deferred_local[drop_mask]
                    new_deferred = deferred_local[defer_mask]
            else:
                keep_idx = torch.tensor([], dtype=torch.long, device=device)
                new_deferred = deferred_local

            cur_seq_len = hidden_states.shape[1]
            all_idx = torch.arange(cur_seq_len, device=device)
            is_candidate = torch.zeros(cur_seq_len, dtype=torch.bool, device=device)
            is_candidate[deferred_local] = True
            non_candidate_idx = all_idx[~is_candidate]
            surviving = torch.sort(torch.cat([non_candidate_idx, keep_idx, new_deferred])).values
            remap = torch.full((cur_seq_len,), -1, dtype=torch.long, device=device)
            remap[surviving] = torch.arange(surviving.numel(), device=device)

            hidden_states = hidden_states[:, surviving, :]
            deferred_local = remap[new_deferred] if new_deferred.numel() > 0 else new_deferred

            # Prune THIS layer's just-written cache to match survivors — required so this
            # layer's cache length (and every later layer's, since they receive the shorter
            # sequence) correctly reflects the post-prune coordinate space going forward.
            past_key_values.layers[i].keys = past_key_values.layers[i].keys[:, :, surviving, :]
            past_key_values.layers[i].values = past_key_values.layers[i].values[:, :, surviving, :]
        else:
            hidden_states = decoder_layer(
                hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings, position_ids=position_ids,
                past_key_values=past_key_values, use_cache=True,
            )

    hidden_states = lm.norm(hidden_states)
    logits = model.language_model.lm_head(hidden_states[:, -1:, :])
    next_token = logits.argmax(dim=-1)
    generated_ids = [next_token.item()]

    # --- Step 3: decode remaining tokens, per-layer cache lengths now differ by segment ---
    for step in range(max_new_tokens - 1):
        if next_token.item() == tokenizer.eos_token_id:
            break
        hidden_states = model.language_model.get_input_embeddings()(next_token)  # [1,1,hidden]

        for i, decoder_layer in enumerate(layers[: config.num_hidden_layers]):
            seg_cache_len = past_key_values.get_seq_length(layer_idx=i)  # THIS layer's own cache length
            cache_position = torch.arange(seg_cache_len, seg_cache_len + 1, device=device)
            attn_mask_1d = torch.ones(1, seg_cache_len + 1, device=device, dtype=torch.long)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(
                hidden_states, attn_mask_1d, cache_position, past_key_values
            )
            hidden_states = decoder_layer(
                hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                position_embeddings=position_embeddings, position_ids=position_ids,
                past_key_values=past_key_values, use_cache=True,
            )

        hidden_states = lm.norm(hidden_states)
        logits = model.language_model.lm_head(hidden_states[:, -1:, :])
        next_token = logits.argmax(dim=-1)
        generated_ids.append(next_token.item())

    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response, len(generated_ids)

In [ ]:
door_sample = next(s for s in TEST_SAMPLES if s["image_file"] == "554523026.jpg")
question_text = f"{door_sample['question']} {VQA_PROMPT_SUFFIX}"
pixel_values = load_image(os.path.join(IMG_DIR, door_sample["image_file"]), max_num=6).to(torch.float16).cuda()

response, n_tokens = dap_generate(model, tokenizer, pixel_values, question_text, max_new_tokens=32)
print("Question:", door_sample["question"])
print("Ground truth:", door_sample["answers"])
print("Full DAP generation:", repr(response))
print("Tokens generated:", n_tokens)
print("VQA accuracy:", vqa_accuracy(response, door_sample["answers"]))

In [ ]:
#27
import gc

full_gen_results = []

for s in TEST_SAMPLES:
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()

    generation_config = dict(max_new_tokens=32, do_sample=False)
    with torch.no_grad():
        unpruned_text = model.chat(tokenizer, pixel_values, question_text, generation_config).strip()

    dap_text, n_tok = dap_generate(model, tokenizer, pixel_values, question_text, max_new_tokens=32)

    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    dap_acc = vqa_accuracy(dap_text, s["answers"])

    full_gen_results.append({
        "file": fname, "question": s["question"],
        "unpruned_text": unpruned_text, "dap_text": dap_text,
        "unpruned_acc": unpruned_acc, "dap_acc": dap_acc,
    })

    # --- Explicit cleanup: free this sample's tensors before the next iteration ---
    del pixel_values
    gc.collect()
    torch.cuda.empty_cache()
    mem_now = torch.cuda.memory_allocated(0) / 1e9
    print(f"{fname}: unpruned='{unpruned_text}' dap='{dap_text}' | GPU mem after cleanup: {mem_now:.2f} GB")

n = len(full_gen_results)
avg_unpruned = sum(r['unpruned_acc'] for r in full_gen_results) / n
avg_dap = sum(r['dap_acc'] for r in full_gen_results) / n

print(f"\nn = {n}")
print(f"Unpruned (real generation) accuracy: {avg_unpruned:.4f}")
print(f"Full DAP (real generation) accuracy: {avg_dap:.4f}")
print(f"Delta: {avg_dap - avg_unpruned:+.4f}")

print("\nAll disagreements:")
for r in full_gen_results:
    if r['unpruned_text'] != r['dap_text']:
        print(f"  {r['file']}: Q='{r['question']}'")
        print(f"    unpruned='{r['unpruned_text']}' (acc={r['unpruned_acc']:.2f})  dap='{r['dap_text']}' (acc={r['dap_acc']:.2f})")

In [ ]:
print("Current attn_implementation:", model.language_model.config._attn_implementation)

# If this prints "eager", that confirms the hypothesis — force it to sdpa explicitly,
# not just by omission, for every non-checkpoint layer's benefit
model.language_model.config._attn_implementation = "sdpa"
print("After override:", model.language_model.config._attn_implementation)

In [ ]:
#A
# === Imports ===
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import types
import json
import gc
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb, eager_attention_forward

# === HF token ===
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# === Load model (NO global eager) ===
MODEL_ID = "OpenGVLab/InternVL3_5-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, quantization_config=bnb_config,
    low_cpu_mem_usage=True, trust_remote_code=True, device_map={"": 0},
).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
layers = model.language_model.model.layers

# Force SDPA globally — checkpoint layers bypass this via direct eager_attention_forward calls
model.language_model.config._attn_implementation = "sdpa"
print("attn_implementation:", model.language_model.config._attn_implementation)
print("GPU memory after load (GB):", torch.cuda.memory_allocated(0) / 1e9)

# === Image preprocessing helpers ===
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_image(image_path, input_size=448, max_num=6):
    image = Image.open(image_path).convert("RGB")
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(img) for img in images]
    return torch.stack(pixel_values)

# === Data paths + samples ===
DATA_DIR = "/kaggle/input/notebooks/mmrafi14/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)
TEST_SAMPLES = SAMPLES[:40]
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."

def vqa_accuracy(predicted: str, ground_truth_answers: list) -> float:
    import re
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for article in ["a ", "an ", "the "]:
            if s.startswith(article):
                s = s[len(article):]
        return s.strip()
    pred_norm = normalise(predicted)
    matches = sum(1 for a in ground_truth_answers if normalise(a) == pred_norm)
    return min(matches / 3, 1.0)

# === DAP scoring + generation ===
CHECKPOINT_LAYERS = [10, 14, 18]
KEEP_PCTILE = 70
DROP_PCTILE = 30

def compute_mean_importance(attn_weights, visual_indices):
    device = attn_weights.device
    seq_len = attn_weights.shape[-1]
    aw = attn_weights[0].mean(dim=0)
    seq_range = torch.arange(seq_len, device=device)
    valid_counts = seq_len - seq_range
    col_sums = aw.sum(dim=0)
    importance_all = col_sums / valid_counts.clamp(min=1).to(col_sums.dtype)
    return importance_all[visual_indices.to(device)]

def dap_generate(model, tokenizer, pixel_values, question, max_new_tokens=32,
                  checkpoint_layers=CHECKPOINT_LAYERS, keep_pctile=KEEP_PCTILE, drop_pctile=DROP_PCTILE):
    device = next(model.parameters()).device
    lm = model.language_model.model
    layers_ = lm.layers
    config = lm.config

    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        raise RuntimeError("OK1")
    orig_lm_gen = model.language_model.generate
    model.language_model.generate = spy_lm_generate
    probe_config = dict(max_new_tokens=1, do_sample=False)
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = orig_lm_gen

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    orig_model_gen = model.generate
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = orig_model_gen

    inputs_embeds = intercepted_local['inputs_embeds']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == model.img_context_token_id).squeeze(0)
    visual_indices = torch.nonzero(vis_mask, as_tuple=True)[0].to(device)

    def build_mask_pos(h, attn_mask_1d, cache_position, past_kv):
        pos_ids = cache_position.unsqueeze(0)
        mask_kwargs = {
            "config": config, "input_embeds": h, "attention_mask": attn_mask_1d,
            "cache_position": cache_position, "past_key_values": past_kv, "position_ids": pos_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        pos_emb = lm.rotary_emb(h, pos_ids)
        return causal_mask_mapping, pos_emb, pos_ids

    past_key_values = DynamicCache(config=config)
    hidden_states = inputs_embeds
    deferred_local = visual_indices.clone()

    with torch.no_grad():
        for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
            cur_len = hidden_states.shape[1]
            cache_position = torch.arange(0, cur_len, device=device)
            attn_mask_1d = torch.ones(1, cur_len, device=device, dtype=torch.long)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(hidden_states, attn_mask_1d, cache_position, past_key_values)
    
            if i in checkpoint_layers:
                is_final = (i == checkpoint_layers[-1])
                attn_module = decoder_layer.self_attn
                original_fwd = attn_module.forward
                local_capture = {}
    
                def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                    input_shape = hidden_states.shape[:-1]
                    hidden_shape = (*input_shape, -1, self.head_dim)
                    q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                    cos, sin = position_embeddings
                    q, k = apply_rotary_pos_emb(q, k, cos, sin)
                    if past_key_values is not None:
                        k, v = past_key_values.update(k, v, self.layer_idx)
                    attn_output, attn_weights = eager_attention_forward(self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling)
                    local_capture['weights'] = attn_weights.detach()
                    attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                    attn_output = self.o_proj(attn_output)
                    return attn_output, None
    
                attn_module.forward = types.MethodType(scoring_forward, attn_module)
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
                attn_module.forward = original_fwd
    
                if deferred_local.numel() > 0:
                    importance = compute_mean_importance(local_capture['weights'], deferred_local)
                    if is_final:
                        median_cut = torch.median(importance.float())
                        keep_mask = importance >= median_cut
                        keep_idx = deferred_local[keep_mask]
                        new_deferred = torch.tensor([], dtype=torch.long, device=device)
                    else:
                        keep_cut = torch.quantile(importance.float(), keep_pctile / 100.0)
                        drop_cut = torch.quantile(importance.float(), drop_pctile / 100.0)
                        keep_mask = importance >= keep_cut
                        drop_mask = (importance <= drop_cut) & (~keep_mask)
                        defer_mask = (~keep_mask) & (~drop_mask)
                        keep_idx = deferred_local[keep_mask]
                        new_deferred = deferred_local[defer_mask]
                else:
                    keep_idx = torch.tensor([], dtype=torch.long, device=device)
                    new_deferred = deferred_local
    
                cur_seq_len = hidden_states.shape[1]
                all_idx = torch.arange(cur_seq_len, device=device)
                is_candidate = torch.zeros(cur_seq_len, dtype=torch.bool, device=device)
                is_candidate[deferred_local] = True
                non_candidate_idx = all_idx[~is_candidate]
                surviving = torch.sort(torch.cat([non_candidate_idx, keep_idx, new_deferred])).values
                remap = torch.full((cur_seq_len,), -1, dtype=torch.long, device=device)
                remap[surviving] = torch.arange(surviving.numel(), device=device)
    
                hidden_states = hidden_states[:, surviving, :]
                deferred_local = remap[new_deferred] if new_deferred.numel() > 0 else new_deferred
    
                past_key_values.layers[i].keys = past_key_values.layers[i].keys[:, :, surviving, :]
                past_key_values.layers[i].values = past_key_values.layers[i].values[:, :, surviving, :]
            else:
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
    
        hidden_states = lm.norm(hidden_states)
        logits = model.language_model.lm_head(hidden_states[:, -1:, :])
        next_token = logits.argmax(dim=-1)
        generated_ids = [next_token.item()]
    
        for step in range(max_new_tokens - 1):
            if next_token.item() == tokenizer.eos_token_id:
                break
            hidden_states = model.language_model.get_input_embeddings()(next_token)
            for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
                seg_cache_len = past_key_values.get_seq_length(layer_idx=i)
                cache_position = torch.arange(seg_cache_len, seg_cache_len + 1, device=device)
                attn_mask_1d = torch.ones(1, seg_cache_len + 1, device=device, dtype=torch.long)
                causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(hidden_states, attn_mask_1d, cache_position, past_key_values)
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
            hidden_states = lm.norm(hidden_states)
            logits = model.language_model.lm_head(hidden_states[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            generated_ids.append(next_token.item())
    
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return response, len(generated_ids)

print("Setup complete.")

In [ ]:
import gc

full_gen_results = []

for idx, s in enumerate(TEST_SAMPLES):
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()

    generation_config = dict(max_new_tokens=32, do_sample=False)
    with torch.no_grad():
        unpruned_text = model.chat(tokenizer, pixel_values, question_text, generation_config).strip()

    dap_text, n_tok = dap_generate(model, tokenizer, pixel_values, question_text, max_new_tokens=32)

    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    dap_acc = vqa_accuracy(dap_text, s["answers"])

    full_gen_results.append({
        "file": fname, "question": s["question"],
        "unpruned_text": unpruned_text, "dap_text": dap_text,
        "unpruned_acc": unpruned_acc, "dap_acc": dap_acc,
    })

    del pixel_values
    gc.collect()
    torch.cuda.empty_cache()
    mem_now = torch.cuda.memory_allocated(0) / 1e9
    mem_reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"[{idx+1}/40] {fname}: unpruned='{unpruned_text}' dap='{dap_text}' | allocated={mem_now:.2f}GB reserved={mem_reserved:.2f}GB")

n = len(full_gen_results)
avg_unpruned = sum(r['unpruned_acc'] for r in full_gen_results) / n
avg_dap = sum(r['dap_acc'] for r in full_gen_results) / n

print(f"\nn = {n}")
print(f"Unpruned (real generation) accuracy: {avg_unpruned:.4f}")
print(f"Full DAP (real generation) accuracy: {avg_dap:.4f}")
print(f"Delta: {avg_dap - avg_unpruned:+.4f}")

print("\nAll disagreements:")
for r in full_gen_results:
    if r['unpruned_text'] != r['dap_text']:
        print(f"  {r['file']}: Q='{r['question']}'")
        print(f"    unpruned='{r['unpruned_text']}' (acc={r['unpruned_acc']:.2f})  dap='{r['dap_text']}' (acc={r['dap_acc']:.2f})")

In [ ]:
#29
import time
from zeus.monitor import ZeusMonitor
monitor = ZeusMonitor(gpu_indices=[0])

print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")

In [ ]:
#30
import time

N_ENERGY_TEST = 10  # dry-run scale before committing to the full 40+ or 1000-sample run

energy_results = []

for s in TEST_SAMPLES[:N_ENERGY_TEST]:
    fname = s["image_file"]
    question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
    pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=6).to(torch.float16).cuda()

    # --- Unpruned, through the SAME custom loop (checkpoint_layers=[]) ---
    monitor.begin_window(f"unpruned_{fname}")
    t0 = time.perf_counter()
    unpruned_text, unpruned_tok_count = dap_generate(
        model, tokenizer, pixel_values, question_text, max_new_tokens=32,
        checkpoint_layers=[]  # <-- disables pruning entirely, same code path otherwise
    )
    unpruned_elapsed = time.perf_counter() - t0
    unpruned_measurement = monitor.end_window(f"unpruned_{fname}")

    # --- Full DAP, same harness, pruning enabled ---
    monitor.begin_window(f"dap_{fname}")
    t0 = time.perf_counter()
    dap_text, dap_tok_count = dap_generate(
        model, tokenizer, pixel_values, question_text, max_new_tokens=32,
        checkpoint_layers=CHECKPOINT_LAYERS
    )
    dap_elapsed = time.perf_counter() - t0
    dap_measurement = monitor.end_window(f"dap_{fname}")

    unpruned_energy_gross = unpruned_measurement.total_energy
    dap_energy_gross = dap_measurement.total_energy
    unpruned_avg_power = unpruned_energy_gross / unpruned_elapsed if unpruned_elapsed > 0 else 0
    dap_avg_power = dap_energy_gross / dap_elapsed if dap_elapsed > 0 else 0
    unpruned_energy_net = max((unpruned_avg_power - IDLE_POWER_W) * unpruned_elapsed, 0)
    dap_energy_net = max((dap_avg_power - IDLE_POWER_W) * dap_elapsed, 0)

    unpruned_acc = vqa_accuracy(unpruned_text, s["answers"])
    dap_acc = vqa_accuracy(dap_text, s["answers"])

    energy_results.append({
        "file": fname,
        "unpruned_energy_gross": unpruned_energy_gross, "dap_energy_gross": dap_energy_gross,
        "unpruned_energy_net": unpruned_energy_net, "dap_energy_net": dap_energy_net,
        "unpruned_elapsed": unpruned_elapsed, "dap_elapsed": dap_elapsed,
        "unpruned_tok": unpruned_tok_count, "dap_tok": dap_tok_count,
        "unpruned_acc": unpruned_acc, "dap_acc": dap_acc,
    })

    print(f"{fname}: unpruned={unpruned_energy_gross:.2f}J/{unpruned_elapsed:.2f}s ({unpruned_tok_count}tok) | "
          f"dap={dap_energy_gross:.2f}J/{dap_elapsed:.2f}s ({dap_tok_count}tok) | "
          f"acc: unpruned={unpruned_acc:.2f} dap={dap_acc:.2f}")

    del pixel_values
    gc.collect()
    torch.cuda.empty_cache()

n = len(energy_results)
avg_unpruned_gross = sum(r['unpruned_energy_gross'] for r in energy_results) / n
avg_dap_gross = sum(r['dap_energy_gross'] for r in energy_results) / n
avg_unpruned_net = sum(r['unpruned_energy_net'] for r in energy_results) / n
avg_dap_net = sum(r['dap_energy_net'] for r in energy_results) / n

print(f"\n--- n={n} summary ---")
print(f"Avg unpruned energy (gross): {avg_unpruned_gross:.2f} J | (net): {avg_unpruned_net:.2f} J")
print(f"Avg DAP energy (gross):      {avg_dap_gross:.2f} J | (net): {avg_dap_net:.2f} J")
print(f"Gross energy reduction: {(1 - avg_dap_gross/avg_unpruned_gross)*100:+.1f}%")
print(f"Net energy reduction:   {(1 - avg_dap_net/avg_unpruned_net)*100:+.1f}%")

In [ ]:
#31
MODEL_NAME = "internvl3.5-8b-dap"
OUT_DIR = "/kaggle/working"
LOG_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_telemetry.csv")
MANIFEST_FILE = os.path.join(OUT_DIR, f"{MODEL_NAME}_manifest.json")

N_RUNS = 5
MAX_NEW_TOKENS = 32
VQA_PROMPT_SUFFIX = "Answer the question using a single word or phrase."
MAX_TILES = 6
CHECKPOINT_LAYERS = [10, 14, 18]
KEEP_PCTILE = 70
DROP_PCTILE = 30

print(f"Log file: {LOG_FILE}")
print(f"N_RUNS: {N_RUNS}")
print(f"CHECKPOINT_LAYERS: {CHECKPOINT_LAYERS}")

In [ ]:
#32
def dap_generate(model, tokenizer, pixel_values, question, max_new_tokens=32,
                  checkpoint_layers=CHECKPOINT_LAYERS, keep_pctile=KEEP_PCTILE, drop_pctile=DROP_PCTILE):
    device = next(model.parameters()).device
    lm = model.language_model.model
    layers_ = lm.layers
    config = lm.config

    intercepted_local = {}
    def spy_lm_generate(*args, **kwargs):
        intercepted_local['inputs_embeds'] = kwargs.get('inputs_embeds')
        raise RuntimeError("OK1")
    orig_lm_gen = model.language_model.generate
    model.language_model.generate = spy_lm_generate
    probe_config = dict(max_new_tokens=1, do_sample=False)
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK1" not in str(e): raise
    model.language_model.generate = orig_lm_gen

    intercepted_ids_local = {}
    def spy_model_gen(*args, **kwargs):
        intercepted_ids_local['input_ids'] = kwargs.get('input_ids')
        raise RuntimeError("OK2")
    orig_model_gen = model.generate
    model.generate = spy_model_gen
    try:
        with torch.no_grad():
            _ = model.chat(tokenizer, pixel_values, question, probe_config)
    except RuntimeError as e:
        if "OK2" not in str(e): raise
    model.generate = orig_model_gen

    inputs_embeds = intercepted_local['inputs_embeds']
    ids = intercepted_ids_local['input_ids']
    vis_mask = (ids == model.img_context_token_id).squeeze(0)
    visual_indices = torch.nonzero(vis_mask, as_tuple=True)[0].to(device)

    def build_mask_pos(h, attn_mask_1d, cache_position, past_kv):
        pos_ids = cache_position.unsqueeze(0)
        mask_kwargs = {
            "config": config, "input_embeds": h, "attention_mask": attn_mask_1d,
            "cache_position": cache_position, "past_key_values": past_kv, "position_ids": pos_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        pos_emb = lm.rotary_emb(h, pos_ids)
        return causal_mask_mapping, pos_emb, pos_ids

    log = {"n_visual_tokens": visual_indices.numel(), "checkpoints": []}

    with torch.no_grad():
        past_key_values = DynamicCache(config=config)
        hidden_states = inputs_embeds
        deferred_local = visual_indices.clone()

        for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
            cur_len = hidden_states.shape[1]
            cache_position = torch.arange(0, cur_len, device=device)
            attn_mask_1d = torch.ones(1, cur_len, device=device, dtype=torch.long)
            causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(hidden_states, attn_mask_1d, cache_position, past_key_values)

            if i in checkpoint_layers:
                is_final = (i == checkpoint_layers[-1])
                attn_module = decoder_layer.self_attn
                original_fwd = attn_module.forward
                local_capture = {}

                def scoring_forward(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
                    input_shape = hidden_states.shape[:-1]
                    hidden_shape = (*input_shape, -1, self.head_dim)
                    q = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    k = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
                    v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
                    cos, sin = position_embeddings
                    q, k = apply_rotary_pos_emb(q, k, cos, sin)
                    if past_key_values is not None:
                        k, v = past_key_values.update(k, v, self.layer_idx)
                    attn_output, attn_weights = eager_attention_forward(self, q, k, v, attention_mask, dropout=0.0, scaling=self.scaling)
                    local_capture['weights'] = attn_weights.detach()
                    attn_output = attn_output.reshape(*input_shape, -1).contiguous()
                    attn_output = self.o_proj(attn_output)
                    return attn_output, None

                attn_module.forward = types.MethodType(scoring_forward, attn_module)
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
                attn_module.forward = original_fwd

                if deferred_local.numel() > 0:
                    importance = compute_mean_importance(local_capture['weights'], deferred_local)
                    if is_final:
                        median_cut = torch.median(importance.float())
                        keep_mask = importance >= median_cut
                        keep_idx = deferred_local[keep_mask]
                        drop_idx = deferred_local[~keep_mask]
                        new_deferred = torch.tensor([], dtype=torch.long, device=device)
                    else:
                        keep_cut = torch.quantile(importance.float(), keep_pctile / 100.0)
                        drop_cut = torch.quantile(importance.float(), drop_pctile / 100.0)
                        keep_mask = importance >= keep_cut
                        drop_mask = (importance <= drop_cut) & (~keep_mask)
                        defer_mask = (~keep_mask) & (~drop_mask)
                        keep_idx = deferred_local[keep_mask]
                        drop_idx = deferred_local[drop_mask]
                        new_deferred = deferred_local[defer_mask]
                else:
                    keep_idx = torch.tensor([], dtype=torch.long, device=device)
                    drop_idx = torch.tensor([], dtype=torch.long, device=device)
                    new_deferred = deferred_local

                log["checkpoints"].append({
                    "layer": i, "candidates": deferred_local.numel(),
                    "kept": keep_idx.numel(), "dropped": drop_idx.numel(), "deferred": new_deferred.numel(),
                })

                cur_seq_len = hidden_states.shape[1]
                all_idx = torch.arange(cur_seq_len, device=device)
                is_candidate = torch.zeros(cur_seq_len, dtype=torch.bool, device=device)
                is_candidate[deferred_local] = True
                non_candidate_idx = all_idx[~is_candidate]
                surviving = torch.sort(torch.cat([non_candidate_idx, keep_idx, new_deferred])).values
                remap = torch.full((cur_seq_len,), -1, dtype=torch.long, device=device)
                remap[surviving] = torch.arange(surviving.numel(), device=device)

                hidden_states = hidden_states[:, surviving, :]
                deferred_local = remap[new_deferred] if new_deferred.numel() > 0 else new_deferred

                past_key_values.layers[i].keys = past_key_values.layers[i].keys[:, :, surviving, :]
                past_key_values.layers[i].values = past_key_values.layers[i].values[:, :, surviving, :]
            else:
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )

        hidden_states = lm.norm(hidden_states)
        logits = model.language_model.lm_head(hidden_states[:, -1:, :])
        next_token = logits.argmax(dim=-1)
        generated_ids = [next_token.item()]

        for step in range(max_new_tokens - 1):
            if next_token.item() == tokenizer.eos_token_id:
                break
            hidden_states = model.language_model.get_input_embeddings()(next_token)
            for i, decoder_layer in enumerate(layers_[: config.num_hidden_layers]):
                seg_cache_len = past_key_values.get_seq_length(layer_idx=i)
                cache_position = torch.arange(seg_cache_len, seg_cache_len + 1, device=device)
                attn_mask_1d = torch.ones(1, seg_cache_len + 1, device=device, dtype=torch.long)
                causal_mask_mapping, position_embeddings, position_ids = build_mask_pos(hidden_states, attn_mask_1d, cache_position, past_key_values)
                hidden_states = decoder_layer(
                    hidden_states, attention_mask=causal_mask_mapping[config.layer_types[i]],
                    position_embeddings=position_embeddings, position_ids=position_ids,
                    past_key_values=past_key_values, use_cache=True,
                )
            hidden_states = lm.norm(hidden_states)
            logits = model.language_model.lm_head(hidden_states[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            generated_ids.append(next_token.item())

    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response, len(generated_ids), log

In [ ]:
#33
def load_manifest():
    if os.path.exists(MANIFEST_FILE):
        with open(MANIFEST_FILE) as f:
            return json.load(f)
    return {"completed_runs": [], "current_run": None}

def save_manifest(manifest):
    with open(MANIFEST_FILE, "w") as f:
        json.dump(manifest, f, indent=2)

manifest = load_manifest()
print("Completed runs so far:", manifest["completed_runs"])

In [ ]:
#34
import time
from zeus.monitor import ZeusMonitor
monitor = ZeusMonitor(gpu_indices=[0])

print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")

In [ ]:
#35
import csv
from datetime import datetime, timezone
write_header = not (os.path.exists(LOG_FILE) and os.path.getsize(LOG_FILE) > 0)
log_f = open(LOG_FILE, "a", newline="", buffering=1)
fieldnames = [
    "run_id", "method", "sample_idx", "question_id", "image_id",
    "question", "predicted", "vqa_accuracy",
    "energy_gross_j", "energy_net_j", "avg_power_w",
    "idle_power_w", "elapsed_s", "tokens_generated",
    "n_visual_tokens", "ckpt1_dropped", "ckpt1_deferred",
    "ckpt2_dropped", "ckpt2_deferred", "ckpt3_dropped",
    "timestamp",
]
writer = csv.DictWriter(log_f, fieldnames=fieldnames)
if write_header:
    writer.writeheader()

for run_id in range(1, N_RUNS + 1):
    if run_id in manifest["completed_runs"]:
        print(f"Run {run_id} already complete — skipping.")
        continue

    print(f"\n{'='*50}\nStarting run {run_id} of {N_RUNS}\n{'='*50}")
    manifest["current_run"] = run_id
    save_manifest(manifest)

    for s in SAMPLES:
        fname = s["image_file"]
        question_text = f"{s['question']} {VQA_PROMPT_SUFFIX}"
        pixel_values = load_image(os.path.join(IMG_DIR, fname), max_num=MAX_TILES).to(torch.float16).cuda()

        for method, ckpt_layers in [("unpruned_matched", []), ("dap", CHECKPOINT_LAYERS)]:
            window_name = f"run{run_id}_{method}_{s['sample_idx']}"
            monitor.begin_window(window_name)
            t0 = time.perf_counter()

            predicted, n_tok, log = dap_generate(
                model, tokenizer, pixel_values, question_text,
                max_new_tokens=MAX_NEW_TOKENS, checkpoint_layers=ckpt_layers
            )

            elapsed = time.perf_counter() - t0
            measurement = monitor.end_window(window_name)

            energy_gross = measurement.total_energy
            avg_power = energy_gross / elapsed if elapsed > 0 else 0
            energy_net = max((avg_power - IDLE_POWER_W) * elapsed, 0)
            acc = vqa_accuracy(predicted, s["answers"])

            ckpts = log["checkpoints"]
            ckpt1_dropped = ckpts[0]["dropped"] if len(ckpts) > 0 else None
            ckpt1_deferred = ckpts[0]["deferred"] if len(ckpts) > 0 else None
            ckpt2_dropped = ckpts[1]["dropped"] if len(ckpts) > 1 else None
            ckpt2_deferred = ckpts[1]["deferred"] if len(ckpts) > 1 else None
            ckpt3_dropped = ckpts[2]["dropped"] if len(ckpts) > 2 else None

            writer.writerow({
                "run_id": run_id, "method": method, "sample_idx": s["sample_idx"],
                "question_id": s["question_id"], "image_id": s["image_id"],
                "question": s["question"], "predicted": predicted,
                "vqa_accuracy": round(acc, 4),
                "energy_gross_j": round(energy_gross, 4),
                "energy_net_j": round(energy_net, 4),
                "avg_power_w": round(avg_power, 3),
                "idle_power_w": round(IDLE_POWER_W, 3),
                "elapsed_s": round(elapsed, 4),
                "tokens_generated": n_tok,
                "n_visual_tokens": log["n_visual_tokens"],
                "ckpt1_dropped": ckpt1_dropped, "ckpt1_deferred": ckpt1_deferred,
                "ckpt2_dropped": ckpt2_dropped, "ckpt2_deferred": ckpt2_deferred,
                "ckpt3_dropped": ckpt3_dropped,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })

        del pixel_values
        gc.collect()
        torch.cuda.empty_cache()

    manifest["completed_runs"].append(run_id)
    manifest["current_run"] = None
    save_manifest(manifest)
    print(f"Run {run_id} complete.")

log_f.close()
print("\nAll runs complete.")

In [ ]:
#36
# Close the file handle if it's still open (interrupt may not have closed it cleanly)
try:
    log_f.close()
    print("Closed log_f")
except Exception as e:
    print(f"log_f already closed or not defined: {e}")

# Inspect what's actually in /kaggle/working right now
print("\nContents of /kaggle/working:")
print(os.listdir("/kaggle/working"))

# Check how far the interrupted run got
if os.path.exists(LOG_FILE):
    import pandas as pd
    df = pd.read_csv(LOG_FILE)
    print(f"\nPartial telemetry file has {len(df)} rows")
    print(df['run_id'].value_counts())
    print(df['method'].value_counts())

if os.path.exists(MANIFEST_FILE):
    with open(MANIFEST_FILE) as f:
        print("\nManifest contents:", json.load(f))

In [ ]:
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        lines = f.readlines()
    print(f"Total lines in file: {len(lines)}")
    print("First line (header):", lines[0] if lines else "EMPTY FILE")
    if len(lines) > 1:
        print("Second line (first data row):", lines[1])
else:
    print("LOG_FILE doesn't exist")

print("\nAll files in /kaggle/working:")
print(os.listdir("/kaggle/working"))

In [ ]:
# Delete the broken (headerless) file and its manifest — this was an accidental
# partial full-scale run anyway, not the n=100 dry run we actually want next
for f in [LOG_FILE, MANIFEST_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

print("\nContents of /kaggle/working now:")
print(os.listdir("/kaggle/working"))